In [4]:
# ==== 1. Imports ====
import requests
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
import time


In [5]:
import os# ==== 2. Config ====
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama:11434")
QDRANT_URL = os.getenv("QDRANT_URL", "http://qdrant:6333")

COLLECTION = os.getenv("QDRANT_COLLECTION", "ti_chunks")

EMBED_MODEL = os.getenv("EMBED_MODEL", "bge-m3:latest")   # или nomic-embed-text
LLM_MODEL   = os.getenv("LLM_MODEL",   "qwen2.5:7b-instruct")  

print("Ready")


Ready


In [6]:
# ==== 3. State ====
class State(TypedDict, total=False):
    query: str
    route: str
    q_embedding: List[float]
    hits: List[Dict[str, Any]]
    context: str
    answer: str
    citations: List[Dict[str, Any]]


In [7]:
# ==== 4. Graph Nodes ====

def route_query(state: State) -> State:
    q = state["query"].strip()
    state["route"] = "rag" if len(q) > 20 else "direct"
    return state

def direct_generate(state: State) -> State:
    r = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={"model": LLM_MODEL, "prompt": state["query"], "stream": False}
    )
    r.raise_for_status()
    state["answer"] = r.json()["response"]
    state["citations"] = []
    return state

def embed_query(state: State) -> State:
    r = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": EMBED_MODEL, "prompt": state["query"]}
    )
    r.raise_for_status()
    state["q_embedding"] = r.json()["embedding"]
    return state

def retrieve(state: State) -> State:
    r = requests.post(
        f"{QDRANT_URL}/collections/{COLLECTION}/points/search",
        json={
            "vector": state["q_embedding"],
            "limit": 8,
            "with_payload": True
        }
    )
    r.raise_for_status()
    state["hits"] = r.json()["result"]
    return state

def build_context(state: State) -> State:
    parts = []
    cites = []
    for h in state.get("hits", [])[:6]:
        p = h.get("payload") or {}
        text = p.get("text", "")
        source = p.get("source", "unknown")
        chunk_id = p.get("chunk_id", h.get("id"))
        if text:
            parts.append(f"[{source}#{chunk_id}]\n{text}")
            cites.append({"source": source, "chunk_id": chunk_id, "score": h.get("score")})
    state["context"] = "\n\n---\n\n".join(parts)
    state["citations"] = cites
    return state

def generate(state: State) -> State:
    prompt = f"""Ты помощник по техдокам.
Используй только контекст.
Если нет ответа — скажи об этом.
Ссылки в формате [source#chunk_id].

Вопрос:
{state['query']}

Контекст:
{state.get('context','')}

Ответ:
"""

    r = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={"model": LLM_MODEL, "prompt": prompt, "stream": False}
    )
    r.raise_for_status()
    state["answer"] = r.json()["response"]
    return state


In [8]:
# ==== 5. Build Graph ====

def decide(state: State) -> str:
    return state.get("route", "rag")

g = StateGraph(State)
g.add_node("route_query", route_query)
g.add_node("direct_generate", direct_generate)
g.add_node("embed_query", embed_query)
g.add_node("retrieve", retrieve)
g.add_node("build_context", build_context)
g.add_node("generate", generate)

g.set_entry_point("route_query")
g.add_conditional_edges("route_query", decide, {
    "direct": "direct_generate",
    "rag": "embed_query"
})

g.add_edge("direct_generate", END)
g.add_edge("embed_query", "retrieve")
g.add_edge("retrieve", "build_context")
g.add_edge("build_context", "generate")
g.add_edge("generate", END)

app = g.compile()
print("Graph ready")


Graph ready


In [9]:
# ==== 6. Test Query ====

question = "Опиши процесс приготовления католита"
start = time.time()
result = app.invoke({"query": question})
elapsed = time.time() - start

print("ANSWER:\n")
print(result["answer"])

print("\nCITATIONS:")
for c in result.get("citations", []):
    print(c)

print("\nTime:", round(elapsed,2), "sec")


ANSWER:

Процесс приготовления католита представляет собой комплекс химических и электрохимических операций. Вот основные шаги:

1. Фильтрация растворов зумпфов основной схемы очистки.

2. Очистка электролита от примесей:
   - Железа
   - Цинка
   - Меди
   - Кобальта и свинца

3. Приготовление карбоната никеля (дополнительный шаг, возможно для определенного этапа очистки).

4. Очистка сточных вод и подготовка стока для отделения утилизации солевого стока.

5. Репульпация первичного железистого кека:

   - Эта операция связана с процессом очистки от железа.
   - Используется репульпатор № 201 для добавления борной кислоты, которая является стабилизирующей добавкой и частью католита.

6. Производство товарного кобальтового концентрата и сырья для кобальтового производства (кобальтовый кек и кобальтовый концентрат).

При этом процесс зависит от взаимосвязанных операций, таких как:

- Очистка электролита от меди с использованием цементатора.
- Фильтрация в пачуке или ёмкости.

Весь процес

In [10]:

for h in result.get("hits", []):
    p = h.get("payload") or {}
    text = p.get("text", "")
    source = p.get("source", "unknown")
    chunk_id = p.get("chunk_id", h.get("id"))
    score = h.get("score")

    print(f"[{source}#{chunk_id}] (score={round(score,4) if score else None})")
    print(text)
    print("\n" + "="*80 + "\n")

print("Time:", round(elapsed, 2), "sec")

[unknown#1] (score=0.6519)
1.1 Сырьё
Исходным сырьём для производства католита является никельсодержащий
раствор Отделения Растворения и Дегазации (ОРиД). Данный раствор имеет
следующий химический состав, кг/м3 (г/дм3):
Ni2+ - не менее 220;
Fe2+ - не более 8,0;
Сu2+ - не более 2,5;
Со2+ - не более 6,0;
Na++К+ не более 8,5;
Cl- -240-260;
SO 2- -55-75;


[unknown#0] (score=0.6464)
Введение
Настоящая технологическая инструкция регламентирует
последовательность технологических операций приготовления никелевого
католита, правила ведения и управления процессом, содержит характеристику
сырья, материалов, оборудования и товарной продукции, методы контроля и
метрологическое обеспечение, а также требования охраны труда и
промышленной безопасности.
Инструкция разработана в соответствии с проектом ФМ.04450-П-ИОС7.1-С
«ЦЭН. ОЭН-2. Электроэкстракция никеля из растворов хлорного растворения
ПНТП на объем производства 145 тыс.т/год электролитного никеля»,
выполненным в 2015 году ООО «Институт Гипроник